# BRGC Revision — Notebook 01: Neural per-query ranks (GPU)

**Purpose.** Re-run the key configurations and save the **per-query filtered ranks** on the test set, which the previous runs did not store. These per-query ranks are what Notebook 02 needs for the **paired significance tests** (Reviewers 2, 3, 4) and the **endpoint-uncertainty grouping analysis** (Reviewer 3, comment 1).

**Configs re-run:** `B0` (endpoints only), `B2` (adaptive one-hop context), `H_ONLY` (head-side context). These are the comparisons the reviewers question (does context help? does dropping tail context hurt?).

**What to attach (right panel -> Add Input):**
1. `brgc-frozen-source` — the patched code (dumps per-query ranks when `KGC_DUMP_PERQUERY=1`).
2. `brgc-kg-splits` — the three datasets.

**Settings:** Accelerator = **GPU T4 x2** (or single T4), Internet = **ON** (first cell downloads DistilBERT once).

**Time budget (Kaggle free: 12h/session, ~30h/week):** run **once per dataset**. CoDEx-S ~1.5h, CoDEx-M ~5h, FB15k-237 ~9-11h (split seeds if needed). Completed runs are skipped on re-run.

In [ ]:
# === GPU + Hugging Face setup (run first) ===
import os, sys, subprocess, importlib.metadata as md, gc
from pathlib import Path
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle T4 GPU (Settings -> Accelerator) before running.")
PINNED = {"transformers":"4.46.3","huggingface_hub":"0.26.5","tokenizers":"0.20.3","accelerate":"0.34.2","safetensors":"0.4.5"}
cur = {}
for p in PINNED:
    try: cur[p]=md.version(p)
    except md.PackageNotFoundError: cur[p]=None
if any(cur[n]!=v for n,v in PINNED.items()):
    subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-cache-dir",*[f"{n}=={v}" for n,v in PINNED.items()]])
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
HF_CACHE_ROOT = Path("/kaggle/working/hf_cache"); HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
HF_LOCAL_MODEL_DIR = Path("/kaggle/working/hf_models/distilbert-base-uncased"); HF_LOCAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"]=str(HF_CACHE_ROOT); os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
snapshot_download(repo_id="distilbert-base-uncased", local_dir=str(HF_LOCAL_MODEL_DIR), cache_dir=str(HF_CACHE_ROOT), max_workers=1)
AutoConfig.from_pretrained(str(HF_LOCAL_MODEL_DIR), local_files_only=True)
AutoTokenizer.from_pretrained(str(HF_LOCAL_MODEL_DIR), use_fast=True, local_files_only=True)
_m = AutoModelForSequenceClassification.from_pretrained(str(HF_LOCAL_MODEL_DIR), local_files_only=True); del _m; gc.collect()
print("DistilBERT snapshot ready. Internet can be turned OFF for the rest.")

In [ ]:
# === Discover attached datasets + stage frozen source and data ===
# ATTACH TWO KAGGLE DATASETS to this notebook (right panel -> Add Input):
#   1) brgc-frozen-source   (contains frozen_source/ with config.py, data_loader.py, model.py, train.py, utils.py)
#   2) brgc-kg-splits       (contains CoDEx-S/, CoDEx-M/, FB15k-237/ each with train.txt valid.txt test.txt)
# The cell below auto-finds them anywhere under /kaggle/input, so the dataset slug names do not matter.
import shutil
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input"); WORK = Path("/kaggle/working"); PROJECT_DIR = WORK/"project"
(PROJECT_DIR/"data").mkdir(parents=True, exist_ok=True)

# find frozen_source (a dir containing all five .py files)
SRC_FILES = ["config.py","data_loader.py","model.py","train.py","utils.py"]
src_dir = None
for p in INPUT_ROOT.rglob("config.py"):
    if all((p.parent/f).exists() for f in SRC_FILES):
        src_dir = p.parent; break
if src_dir is None:
    raise FileNotFoundError("Could not find frozen_source (dir with all 5 .py files). Attach the brgc-frozen-source dataset.")
for f in SRC_FILES: shutil.copy2(src_dir/f, PROJECT_DIR/f)
print("frozen_source staged from:", src_dir)

# find dataset dirs (dir with train/valid/test.txt) and copy under project/data/<NAME>
WANTED = ["CoDEx-S","CoDEx-M","FB15k-237"]
found = {}
for name in WANTED:
    for cand in INPUT_ROOT.rglob(name):
        if cand.is_dir() and all((cand/s).exists() for s in ["train.txt","valid.txt","test.txt"]):
            found[name]=cand; break
for name,cand in found.items():
    dst = PROJECT_DIR/"data"/name; dst.mkdir(parents=True, exist_ok=True)
    for s in ["train.txt","valid.txt","test.txt"]: shutil.copy2(cand/s, dst/s)
    print(f"data staged: {name} <- {cand}")
if not found:
    raise FileNotFoundError("No dataset folders found. Attach brgc-kg-splits (CoDEx-S/CoDEx-M/FB15k-237).")
print("Datasets available this session:", sorted(found))

In [ ]:
# === CHOOSE THE DATASET FOR THIS SESSION ===
# Run the notebook ONCE PER DATASET (each is a separate Kaggle session within your 12h limit).
# Approx cost on a T4 for B0+B2+H_ONLY x 3 seeds (9 runs): CoDEx-S ~1.5h, CoDEx-M ~5h, FB15k-237 ~9-11h.
# If FB15k-237 risks exceeding 12h, set SEEDS=[42] first, then [13,2025] in a later session.
DATASET_NAME = "CoDEx-S"          # <-- change to "CoDEx-M" or "FB15k-237" for the other sessions
RUN_IDS = ["B0","B2","H_ONLY"]     # the configs the significance tests + endpoint grouping need
SEEDS   = [13, 42, 2025]           # for FB15k-237 you may split into [42] then [13,2025]
assert DATASET_NAME in found, f"{DATASET_NAME} not attached; found {sorted(found)}"
STUDY_ROOT = PROJECT_DIR/"outputs"/DATASET_NAME/"strict_final"; STUDY_ROOT.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATASET_NAME, "| configs:", RUN_IDS, "| seeds:", SEEDS)

import subprocess, sys
cc = subprocess.run([sys.executable,"-m","py_compile",*SRC_FILES], cwd=PROJECT_DIR, capture_output=True, text=True)
if cc.returncode!=0: raise RuntimeError("frozen source compile failed:\n"+cc.stderr)
print("frozen source compiles OK")

In [ ]:
# === Execute: retrain each config and DUMP per-query test ranks ===
import os, sys, subprocess, json, time, shutil
def run_dir_for(run_id, seed): return STUDY_ROOT/run_id/f"seed_{seed}"
log_root = STUDY_ROOT/"logs"; log_root.mkdir(parents=True, exist_ok=True)
n=0
for run_id in RUN_IDS:
    for seed in SEEDS:
        n+=1
        out_dir = run_dir_for(run_id, seed)
        if (out_dir/"per_query_ranks.npz").exists() and (out_dir/"run_complete.json").exists():
            print(f"SKIP done: {run_id} seed={seed}"); continue
        if out_dir.exists(): shutil.rmtree(out_dir)
        env = os.environ.copy()
        env.update({
            "KGC_DATASET": DATASET_NAME,
            "KGC_DATA_ROOT": str(PROJECT_DIR/"data"),
            "KGC_OUTPUT_ROOT": str(PROJECT_DIR/"outputs"),
            "KGC_EXPERIMENT_ID": run_id, "KGC_SEED": str(seed), "KGC_DETERMINISTIC":"1",
            "KGC_NUM_WORKERS":"0", "TOKENIZERS_PARALLELISM":"false",
            "MASTER_PORT": str(29600+n),
            "KGC_MODEL_NAME": str(HF_LOCAL_MODEL_DIR), "KGC_LOCAL_FILES_ONLY":"1",
            "HF_HOME": str(HF_CACHE_ROOT), "HF_HUB_OFFLINE":"1", "TRANSFORMERS_OFFLINE":"1",
            "KGC_DUMP_PERQUERY":"1",   # <-- turns on per-query rank dumping (this notebook only)
        })
        lp = log_root/f"{run_id}_seed{seed}.log"
        print("="*80); print(f"START {run_id} seed={seed} -> {out_dir}"); print("="*80)
        t0=time.time()
        with lp.open("w") as lf:
            pr = subprocess.run([sys.executable,"train.py"], cwd=PROJECT_DIR, env=env, stdout=lf, stderr=subprocess.STDOUT, text=True)
        dt=time.time()-t0
        if pr.returncode!=0:
            print(lp.read_text()[-6000:]); raise RuntimeError(f"Run failed: {run_id} seed={seed}")
        # finalize test_metrics.json from the metrics jsonl
        mp = out_dir/"relation_metrics.jsonl"
        metrics=[json.loads(l) for l in mp.read_text().splitlines() if l.strip()]
        (out_dir/"test_metrics.json").write_text(json.dumps(metrics[-1], indent=2))
        (out_dir/"run_complete.json").write_text(json.dumps({"run":run_id,"seed":seed,"elapsed_min":dt/60}, indent=2))
        assert (out_dir/"per_query_ranks.npz").exists(), "per_query_ranks.npz was not written!"
        print(f"DONE {run_id} seed={seed} | {dt/60:.1f} min | FilteredMRR={metrics[-1].get('FilteredMRR'):.4f} | per-query dump OK")
print("\nAll requested runs complete for", DATASET_NAME)

In [ ]:
# === Package artifacts to download (keep every zip; send them to me) ===
import shutil, json
from pathlib import Path
from IPython.display import FileLink, display
PKG = Path(f"/kaggle/working/BRGC_perquery_{DATASET_NAME}")
if PKG.exists(): shutil.rmtree(PKG)
PKG.mkdir(parents=True)
# copy the test split so I can join ranks -> (head, tail) by row order
shutil.copy2(PROJECT_DIR/"data"/DATASET_NAME/"test.txt", PKG/"test.txt")
for run_id in RUN_IDS:
    for seed in SEEDS:
        src = STUDY_ROOT/run_id/f"seed_{seed}"
        if not src.exists(): continue
        dst = PKG/run_id/f"seed_{seed}"; dst.mkdir(parents=True, exist_ok=True)
        for fn in ["per_query_ranks.npz","test_metrics.json","run_config.json","run_complete.json","selected_checkpoint.json","pair_overlap_audit.json"]:
            if (src/fn).exists(): shutil.copy2(src/fn, dst/fn)
arc = shutil.make_archive(str(PKG), "zip", PKG)
print("Created:", arc); display(FileLink(arc))